## AI Search keyword 검색 구현

In [ ]:
import requests
import dotenv
import os

In [ ]:
# OPEN API KEY 불러오기
dotenv.load_dotenv()
OPEN_AI_KEY = os.getenv("OPEN_AI_KEY")
SEARCH_API_KEY = os.getenv("SEARCH_API_KEY")

In [ ]:
# Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
endpoint = "https://9ai043search2.search.windows.net/indexes/travel-index/docs/search?api-version=2025-11-01-preview"

# Method : Post

# Header: OpenAI API Key 포함된 헤더 정보
headers = {
    "api-key": SEARCH_API_KEY,
    "Content-Type": "application/json"
}
# Body: Body의 메시지 정보
body = { 
        "search": "이순신",
        # "select": "name, description",
        # "searchFields": "description, name, address",
        "top": 10,
        "count": True  
        }

# OpenAI API 호출
response = requests.post(endpoint, headers=headers, json=body)
response_json = response.json()

# 응답에서 id, name, address, description 파싱
count = response_json['@odata.count']
print(f'count: {count}')
value_list = response_json['value']

for item in value_list:
    item_id = item['id']
    name = item['name']
    address = item['address']
    description = item['description']
    has_parkinglot = item['has_parkinglot']
    print(f"id: {item_id}, name: {name}, address: {address}, description: {description}")
print(response)

## AI Search 의미 체계(Semantic) 검색 구현

In [ ]:
# Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
endpoint = "https://9ai043search2.search.windows.net/indexes/travel-index/docs/search?api-version=2025-11-01-preview"

# Method : Post

# Header: OpenAI API Key 포함된 헤더 정보
headers = {
    "api-key": SEARCH_API_KEY,
    "Content-Type": "application/json"
}
# Body: Body의 메시지 정보
body = { 
        "search": "이순신 장군의 위대함을 알 수 있는 곳",
        "select": "name, description",
        "count": True,
        "queryType": "semantic",
        "semanticConfiguration": "travel-semantic",
        "captions": "extractive|highlight-true",
        "answers": "extractive|count-5",
        "top": 10,
        "queryLanguage": "ko-KR"
        }

# OpenAI API 호출
response = requests.post(endpoint, headers=headers, json=body)
response_json = response.json()

# 응답에서 id, name, address, description 파싱
value_list = response_json['value']
answer_list = response_json['@search.answers']

print("================ANSWER================")
for item in answer_list:
    answer_id = item['key']
    text = item['text']
    highlights = item['highlights']
    score = item['score']
    print(f"id: {answer_id}, text: {text}, highlights: {highlights}, score: {score}")

print("================VALUE================")
for item in value_list:
    score = item['@search.score']
    rerankerscore = item['@search.rerankerScore']
    caption_list = item['@search.captions']

    for cap in caption_list:
        highlights = cap['highlights']
    
    name = item['name']
    description_words = item['description'].split()[:5]
    description = " ".join(description_words)

    print(f"score: {score}, rerankerscore: {rerankerscore}, highlights: {highlights}, name:{name}, description: {description}")

print(response)

## AI Search 벡터 검색 구현

## Gradio 연결
https://www.gradio.app/docs/gradio/blocks

In [ ]:
%pip install gradio --upgrade

# % : 해당 환경/가상환경에 설치됨
# ! : global에 설치됨 

In [ ]:
import gradio as gr
gr.__version__

In [ ]:
# 화면 구상해보기
def greet(name):
    return "Hello " + name + "!"

iface = gr.Interface(fn=greet, inputs='text', outputs='text')
iface.launch()

# fn : UI를 wrap할 함수
# inputs :입력에 사용할 gradio 구성 요소. 함수 인수 개수랑 일치해야 함
# outputs : 출력에 사용할 gradio 구성 요소. 함수 리턴 개수랑 일치해야 함

In [ ]:
# ChatInterface 구현해보기
import random
import gradio as gr

def altermatingly_agree(message, history):
    if len(history) % 2 ==0:
        return f"맞아요, 저도 {message}라고 생각해요."
    else:
        return "그건 아닌 것 같아요ㅎㅎ"

gr.ChatInterface(altermatingly_agree).launch()

In [ ]:
with gr.Blocks() as demo:
    input = gr.Textbox(label="Hello World", value="안녕 세상아")

demo.launch(share=True) #72시간 동안 유효, 공유 가능

In [ ]:
def update(name):
    return f'Welcome to Gradio, {name}!'

with gr.Blocks() as demo:
    gr.Markdown("Start typing below and then click **Run** to see the output.")
    with gr.Row():
        inp = gr.Textbox(placeholder="What's your last name?")
        inp2 = gr.Textbox(placeholder="What's your first name?")
        out = gr.Textbox()
    btn = gr.Button("Run")
    btn.click(fn=update, inputs=inp, outputs=inp2)

demo.launch()

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    history = [
    {"role": "assistant", "content": "I am happy to provide you that report and plot."},
    {"role": "user", "content": "여름에 가기 좋은 남해 관광지 추천해줘"},
    {"role"}
    ]

    chatbot = gr.Chatbot(history, label='날짜 관광지 챗봇')

demo.launch(share=True)

### 블록 배치하기

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## 남해 관광지 챗봇")

    with gr.Column():
        chatbot = gr.Chatbot(label='날짜 관광지 챗봇')

        with gr.Row():
            input_textbox = gr.Textbox(label="질문을 입력하세요", scale = 5)
            send_button = gr.Button("전송", scale=1)

demo.launch(share=True)

### openAI에게 요청 보내기

In [ ]:
def request_openai(prompt):
    import requests
    import os 
    import dotenv
    dotenv.load_dotenv()
    OPEN_AI_KEY = os.getenv("OPEN_AI_KEY")

    # Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
    endpoint = "https://9ai043-openai.openai.azure.com/openai/deployments/9ai043-gpt-4o-mini/chat/completions?api-version=2025-01-01-preview"

    # Method : Post

    # Header: OpenAI API Key 포함된 헤더 정보
    headers = {
        # "api-key": OPEN_AI_KEY,
        "Authorization": OPEN_AI_KEY,
        "Content-Type": "application/json"
    }
    # Body: Body의 메시지 정보
    body = { 
            "messages": [ 
                {
                "role": "user",
                "content": prompt
                }
            ],
            "max_tokens": 4096,
            "temperature": 0.7,
            "top_p": 0.95,
            "model": "9ai043-gpt-4o-mini"
            }

    # OpenAI API 호출
    response = requests.post(endpoint, headers=headers, json=body)
    response_json = response.json()

    content = response_json['choices'][0]['message']['content']
    role = response_json['choices'][0]['message']['role']
    print(role, content)
    return {"role": role, "content": content}

with gr.Blocks() as demo:
    gr.Markdown("## 남해 관광지 챗봇")

    def click_send(prompt, histories):
        assistant_data = request_openai(prompt)
        # histories = list()
        histories.append({"role": "user", "content": prompt})
        histories.append(assistant_data)
        print(histories)
        return histories

    with gr.Column():
        chatbot = gr.Chatbot(label='날짜 관광지 챗봇')

        with gr.Row():
            input_textbox = gr.Textbox(label="질문을 입력하세요", scale = 5)
            send_button = gr.Button("전송", scale=1)
    
    send_button.click(fn=click_send, inputs=[input_textbox, chatbot], outputs= [chatbot])

demo.launch(share=True)


### 이전 대화 반영하기

In [ ]:
def request_openai(prompt, histories):
    import requests
    import os 
    import dotenv

    dotenv.load_dotenv()
    OPEN_AI_KEY = os.getenv("OPEN_AI_KEY")

    # Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
    endpoint = "https://9ai043-openai.openai.azure.com/openai/deployments/9ai043-gpt-4o-mini/chat/completions?api-version=2025-01-01-preview"

    # Method : Post

    # Header: OpenAI API Key 포함된 헤더 정보
    headers = {
        # "api-key": OPEN_AI_KEY,
        "Authorization": OPEN_AI_KEY,
        "Content-Type": "application/json"
    }

    ###### 이전 메시지 반영 코드 추가 ######

    messsage_list = list()

    # 시스템 메시지를 메시지 리스트에 추가
    messsage_list.append({
        "role":"system", 
        "content": [{
            "type": "text",
            "text": "## 역할\n당신은 '보물섬 남해'를 전문적으로 안내하는 친절하고 유능한 관광 가이드입니다. 제공된 남해 관광지 데이터를 바탕으로 사용자의 여행 계획을 돕고 정보를 제공합니다.\n\n## 답변 원칙\n1. **데이터 우선주의**: 반드시 제공된 검색 결과(Context)에 있는 정보만을 바탕으로 답변하세요. 데이터에 없는 내용은 추측하여 지어내지 마세요.\n2. **구체적 정보 활용**: 답변 시 관광지명, 주소, 테마, 주차 가능 여부(has_parkinglot) 정보를 적극적으로 언급하여 실질적인 도움을 주세요.\n3. **태도**: 남해의 따뜻하고 아름다운 이미지를 전달할 수 있도록 친절하고 환대하는 말투를 사용하세요.\n4. **모르는 경우**: 데이터에 해당 관광지 정보가 없다면 \"죄송하지만, 현재 제가 가진 남해 관광 데이터에는 해당 장소에 대한 정보가 없습니다.\"라고 정직하게 답변하세요.\n\n## 답변 형식 (예시)\n- 추천 시: \"[관광지명]을 추천해 드려요! 이곳은 [테마] 테마의 장소로, [주소]에 위치해 있습니다. (주차 가능 여부 언급)\"\n- 상세 설명: 데이터의 'description' 내용을 요약하여 흥미롭게 전달하세요."
        }]
    })

    # 히스토리가 존재하는 경우 메시지 리스트에 모든 데이터를 추가
    for history in histories:
        messsage_list.append(history)

    # 사용자 프롬프트 메시지를 메시지 리스트에 추가한다
    messsage_list.append({
        "role":"user", 
        "content": [{
            "type": "text",
            "text": prompt
        }]
    })

    # Body: Body의 메시지 정보
    body = { 
            "messages": messsage_list,
            "max_tokens": 4096,
            "temperature": 0.7,
            "top_p": 0.95,
            "model": "9ai043-gpt-4o-mini"
            }

    # OpenAI API 호출
    response = requests.post(endpoint, headers=headers, json=body)
    response_json = response.json()

    content = response_json['choices'][0]['message']['content']
    role = response_json['choices'][0]['message']['role']
    print(role, content)
    return {"role": role, "content": content}

import gradio as gr
with gr.Blocks() as demo:
    gr.Markdown("## 남해 관광지 챗봇")

    def click_send(prompt, histories):
        # print(histories)
        # assistant_data = request_openai(prompt)
        assistant_data = request_openai(prompt, histories) # 히스토리도 같이 넘겨줌
        # histories = list()
        histories.append({"role": "user", "content": prompt})
        histories.append(assistant_data)
        print(histories)
        return histories

    with gr.Column():
        chatbot = gr.Chatbot(label='날짜 관광지 챗봇')

        with gr.Row():
            input_textbox = gr.Textbox(label="질문을 입력하세요", scale = 5)
            send_button = gr.Button("전송", scale=1)
    
    send_button.click(fn=click_send, inputs=[input_textbox, chatbot], outputs= [chatbot])

demo.launch(share=True)
# history_list = [{'role': 'user', 'metadata': None, 'content': [{'text': '안녕', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '안녕하세요! 어떻게 도와드릴까요?', 'type': 'text'}], 'options': None}, {'role': 'user', 'content': '반가워'}, {'role': 'assistant', 'content': '반가워요! 어떻게 도와드릴까요?'}]
# click_send("남해에서 가볼만한 곳이 어디야?", history_list)

### 벡터 검색 추가하기

In [ ]:
def request_openai(prompt, histories):
    import requests
    import os 
    import dotenv

    dotenv.load_dotenv()
    OPEN_AI_KEY = os.getenv("OPEN_AI_KEY")
    SEARCH_API_KEY = os.getenv("SEARCH_API_KEY")

    # Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
    endpoint = "https://9ai043-openai.openai.azure.com/openai/deployments/9ai043-gpt-4o-mini/chat/completions?api-version=2025-01-01-preview"

    # Method : Post

    # Header: OpenAI API Key 포함된 헤더 정보
    headers = {
        # "api-key": OPEN_AI_KEY,
        "Authorization": OPEN_AI_KEY,
        "Content-Type": "application/json"
    }

    ###### 이전 메시지 반영 코드 추가 ######

    messsage_list = list()

    # 시스템 메시지를 메시지 리스트에 추가
    messsage_list.append({
        "role":"system", 
        # "content": [{
        #     "type": "text",
        #     "text": "## 역할\n당신은 '보물섬 남해'를 전문적으로 안내하는 친절하고 유능한 관광 가이드입니다. 제공된 남해 관광지 데이터를 바탕으로 사용자의 여행 계획을 돕고 정보를 제공합니다.\n\n## 답변 원칙\n1. **데이터 우선주의**: 반드시 제공된 검색 결과(Context)에 있는 정보만을 바탕으로 답변하세요. 데이터에 없는 내용은 추측하여 지어내지 마세요.\n2. **구체적 정보 활용**: 답변 시 관광지명, 주소, 테마, 주차 가능 여부(has_parkinglot) 정보를 적극적으로 언급하여 실질적인 도움을 주세요.\n3. **태도**: 남해의 따뜻하고 아름다운 이미지를 전달할 수 있도록 친절하고 환대하는 말투를 사용하세요.\n4. **모르는 경우**: 데이터에 해당 관광지 정보가 없다면 \"죄송하지만, 현재 제가 가진 남해 관광 데이터에는 해당 장소에 대한 정보가 없습니다.\"라고 정직하게 답변하세요.\n\n## 답변 형식 (예시)\n- 추천 시: \"[관광지명]을 추천해 드려요! 이곳은 [테마] 테마의 장소로, [주소]에 위치해 있습니다. (주차 가능 여부 언급)\"\n- 상세 설명: 데이터의 'description' 내용을 요약하여 흥미롭게 전달하세요."
        # }]
        "content":  "## 역할\n당신은 '보물섬 남해'를 전문적으로 안내하는 친절하고 유능한 관광 가이드입니다. 제공된 남해 관광지 데이터를 바탕으로 사용자의 여행 계획을 돕고 정보를 제공합니다.\n\n## 답변 원칙\n1. **데이터 우선주의**: 반드시 제공된 검색 결과(Context)에 있는 정보만을 바탕으로 답변하세요. 데이터에 없는 내용은 추측하여 지어내지 마세요.\n2. **구체적 정보 활용**: 답변 시 관광지명, 주소, 테마, 주차 가능 여부(has_parkinglot) 정보를 적극적으로 언급하여 실질적인 도움을 주세요.\n3. **태도**: 남해의 따뜻하고 아름다운 이미지를 전달할 수 있도록 친절하고 환대하는 말투를 사용하세요.\n4. **모르는 경우**: 데이터에 해당 관광지 정보가 없다면 \"죄송하지만, 현재 제가 가진 남해 관광 데이터에는 해당 장소에 대한 정보가 없습니다.\"라고 정직하게 답변하세요.\n\n## 답변 형식 (예시)\n- 추천 시: \"[관광지명]을 추천해 드려요! 이곳은 [테마] 테마의 장소로, [주소]에 위치해 있습니다. (주차 가능 여부 언급)\"\n- 상세 설명: 데이터의 'description' 내용을 요약하여 흥미롭게 전달하세요."
    })

    # 히스토리가 존재하는 경우 메시지 리스트에 모든 데이터를 추가
    for history in histories:
        # content가 텍스트가 아닌 경우
        role = history['role']
        content = history['content'][0]['text']
        messsage_list.append({"role": role, "content": content})

        # content가 텍스트인 경우 - 그대로 넣기
        # messsage_list.append(history)
        
    # 사용자 프롬프트 메시지를 메시지 리스트에 추가한다
    messsage_list.append({
        "role":"user", 
        # "content": [{
        #     "type": "text",
        #     "text": prompt
        # }]
        "content": prompt
    })

    # Body: Body의 메시지 정보
    body = {
        "messages": messsage_list,
        "max_tokens": 4096,
        "temperature": 0.7,
        "top_p": 0.95,
        "data_sources": [
            {
                "type": "azure_search",
                "parameters": {
                    "endpoint": "https://9ai043search2.search.windows.net",
                    "index_name": "travel-index-vector",
                    "semantic_configuration": "travel-semantic",
                    "query_type": "vector_semantic_hybrid",
                    # "query_type": "vector",
                    # "query_type": "vector_simple_hybrid",
        
                    "embedding_dependency": {
                        "type": "deployment_name",
                        "deployment_name": "text-embedding-3-small"
                    },
                    "fields_mapping": {
                        "vector_fields": ["description_vector"]
                    },
                    "in_scope": True,
                    "filter": None,
                    "strictness": 3,
                    "top_n_documents": 5,
                    "authentication": {
                        "type": "api_key",
                        "key": SEARCH_API_KEY
                    }
                }
            }
        ]
    }
 

    # OpenAI API 호출
    response = requests.post(endpoint, headers=headers, json=body)
    
    # 예외 처리 추가
    # if response 
    
    response_json = response.json()

    content = response_json['choices'][0]['message']['content']
    role = response_json['choices'][0]['message']['role']
    print(role, content)
    return {"role": role, "content": content}

import gradio as gr
with gr.Blocks() as demo:
    gr.Markdown("## 남해 관광지 챗봇")

    def click_send(prompt, histories):
        # print(histories)
        # assistant_data = request_openai(prompt)
        assistant_data = request_openai(prompt, histories) # 히스토리도 같이 넘겨줌
        # histories = list()
        histories.append({"role": "user", "content": prompt})
        histories.append(assistant_data)
        print(histories)
        return histories

    with gr.Column():
        chatbot = gr.Chatbot(label='날짜 관광지 챗봇')

        with gr.Row():
            input_textbox = gr.Textbox(label="질문을 입력하세요", scale = 5)
            send_button = gr.Button("전송", scale=1)
    
    send_button.click(fn=click_send, inputs=[input_textbox, chatbot], outputs= [chatbot])

demo.launch(share=True)

# 테스트용 코드
# history_list = [
#     {'role': 'user', 'metadata': None, 'content': '안녕', 'options': None},
#     {'role': 'assistant', 'metadata': None, 'content': '안녕하세요!', 'options': None},
#     {'role': 'user', 'content': '반가워'},
#     {'role': 'assistant', 'content': '반가워요!'}
# ]
# click_send("남해에서 가볼만한 곳이 어디야?", history_list)

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://3476ef0523abefb2f3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


assistant 안녕하세요! 남해에 대해 궁금한 점이 있으시면 언제든지 물어보세요. 제가 도와드릴 수 있는 정보를 제공해 드리겠습니다!
[{'role': 'user', 'content': 'ㅎㅇ'}, {'role': 'assistant', 'content': '안녕하세요! 남해에 대해 궁금한 점이 있으시면 언제든지 물어보세요. 제가 도와드릴 수 있는 정보를 제공해 드리겠습니다!'}]
assistant 남해의 아름다운 관광지를 몇 곳 추천해 드릴게요!

1. **남해 대교**
   - **주소**: 경상남도 남해군 남해면 남해대로 179-45
   - **테마**: 다리
   - **주차**: 주차 가능
   - 남해 대교는 남해와 본토를 연결하는 아름다운 다리로, 주변 경관이 매우 아름다워 사진 촬영하기 좋은 장소입니다.

2. **남해 독일마을**
   - **주소**: 경상남도 남해군 남해읍 독일로 679
   - **테마**: 문화마을
   - **주차**: 주차 가능
   - 이곳은 독일의 전통 건축 양식을 가진 마을로, 독일의 문화를 경험할 수 있는 다양한 음식과 기념품을 즐길 수 있습니다.

3. **상주은모래비치**
   - **주소**: 경상남도 남해군 상주면 상주리 17
   - **테마**: 해변
   - **주차**: 주차 가능
   - 상주은모래비치는 맑은 바다와 고운 모래가 매력적인 해변으로, 여름철 수영과 바다 레포츠를 즐기기에 최적의 장소입니다.

4. **남해 바래길**
   - **주소**: 경상남도 남해군 남해읍 바래길
   - **테마**: 트레킹
   - **주차**: 주차 가능
   - 남해 바래길은 해안을 따라 걷는 트레킹 코스로, 아름다운 자연 경관을 감상하며 산책할 수 있는 코스입니다.

각각의 장소는 남해의 매력을 한껏 느낄 수 있는 곳들이니, 꼭 방문해 보시길 추천드립니다!
[{'role': 'user', 'metadata': None, 'content': [{'text': 'ㅎㅇ', 'type': 'text'}